In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from zapbench.ts_forecasting import util

plt.style.use("science")

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='bottom', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

path_to_inference = glob.glob("/mnt/storage/misc/inference/linear/subject_01/**/", recursive=True)[-1]
df = util.get_per_step_metrics_from_directory(path_to_inference,metric='MAE')
df.sort_values('condition')

In [ ]:
for condition in df["condition"].unique():
  print(df[df["condition"] == condition]['MAE'].mean())

In [ ]:
subject_ids = ["12"]
model_names = ["mean", "linear"]
steps_ahead = [1, 4, 8, 16, 32]
performance_dict = {}
for subject_id in subject_ids:
  performance_dict[subject_id] = {}
  for model_name in model_names:
    performance_dict[subject_id][model_name] = {}
    for step_ahead in steps_ahead:
      path_to_inference = glob.glob(f"/mnt/storage/misc/inference/{model_name}/subject_{subject_id}/**/", recursive=True)[-1]
      print(path_to_inference)
      df = util.get_per_step_metrics_from_directory(path_to_inference, metric='MAE')
      performance_dict[subject_id][model_name][step_ahead] = df['MAE'][df['steps_ahead'] == step_ahead].mean()
performance_dict


In [ ]:
subject_id = "12"
performance_dict[subject_id]
fig, ax = plt.subplots(1, 1, figsize=(2, 2), dpi=200)
for model_name in model_names:
  ax.plot(performance_dict[subject_id][model_name].keys(), performance_dict[subject_id][model_name].values(), '.', label=model_name)
ax.legend()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(4, 2), dpi=500)

subject_ids = list(performance_dict.keys())
model_names = list(next(iter(performance_dict.values())).keys())
colors = ['k', 'tab:blue', 'tab:orange', 'tab:green', 'tab:red']
markers = ['o', 's', 'D', '^', 'v']

for i, model_name in enumerate(model_names):
    performances = [performance_dict[sid][model_name] for sid in subject_ids]
    ax1.plot(
        range(len(subject_ids)),
        performances,
        marker=markers[i % len(markers)],
        color=colors[i % len(colors)],
        linestyle='-',
        linewidth=1.5,
        markersize=6,
        label=model_name
    )

ax1.set_ylabel("$G_x$", fontsize=12, labelpad=-10)
ax1.yaxis.label.set_position((0.0, 0.5))

# Set x-ticks to subject ids
ax1.set_xticks(range(len(subject_ids)))
ax1.set_xticklabels(subject_ids, fontsize=12)

# Optionally, set y-limits to a reasonable range for MAE
all_performances = [
    performance_dict[sid][model_name]
    for sid in subject_ids
    for model_name in model_names
]
ax1.set_ylim(0, max(all_performances)*1.1)

# Style
for spine in ax1.spines.values():
    spine.set_linewidth(1.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(which='minor', length=0)
ax1.tick_params(axis='both', labelsize=12)
ax1.tick_params(axis='x', which='major', top=False, direction="out", width=1.2, length=4)
ax1.tick_params(axis='y', which='major', right=False, direction="out", width=1.2, length=4)

ax1.legend(title="Model", fontsize=10, title_fontsize=10, loc='upper left', frameon=False)

plt.tight_layout()
plt.show()